# BTC 25-Minute Price Predictor — Snowflake offline training

Runs entirely **without internet**. The dataset ships in the repo as Parquet
shards; only `pip install` reaches outside.

Target environment: Snowflake Container Runtime, **4× A10 (23 GB)**, 48 vCPU, 100 GB RAM.

**Order:** run the cells top to bottom. Step 2 is the long one.


In [ ]:
# 1. Environment and hardware audit
!pip install -q -r requirements.txt
!PYTHONPATH=src python3 -m btcpred.utils.hardware

In [ ]:
## 1b. Confirm the dataset is present (no download happens here)
!ls -la data/btcusdt_1s | head -20
!echo "shards: $(ls data/btcusdt_1s/*.parquet | wc -l)"
!du -sh data/btcusdt_1s

In [ ]:
# 2. Build the memory-mapped feature cache (one-off, ~10 min for full history)
# Add --start 2025-01 to build a smaller cache first if you want a fast loop.
!python3 scripts/build_cache.py --shards data/btcusdt_1s --cache data/cache

In [ ]:
# 3. Train the ensemble: 4 predictor variants, one per GPU, one shared CPU analyser
#
# There are no epochs. --max-steps counts *market seconds* replayed.
# 400k steps at batch 16/rank is ~6.4M stream-seconds per variant.
#
# Set ANALYSER_DEVICE=cuda for ~100x faster steps (see README, "Throughput").
!STEPS=400000 MODE=ensemble ANALYSER_DEVICE=cpu bash scripts/snowflake_train.sh

In [ ]:
# 4. Inspect the scoreboard
import json, pathlib
print(pathlib.Path("models/selection.json").read_text())

In [ ]:
# 5. Plot the training curves (offline, matplotlib only)
import json, glob
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for path in sorted(glob.glob("runs/ensemble/log_rank*.jsonl")):
    rows = [json.loads(l) for l in open(path) if '"split"' not in l]
    name = rows[0]["variant"] if rows else path
    steps = [r["step"] for r in rows]
    axes[0].plot(steps, [r.get("mae_bps/terminal") for r in rows], label=name)
    axes[1].plot(steps, [r.get("dir_acc") for r in rows], label=name)
    axes[2].plot(steps, [r.get("stability_bps") for r in rows], label=name)
for ax, t in zip(axes, ["terminal MAE (bps)", "directional accuracy", "stability (bps/tick)"]):
    ax.set_title(t); ax.set_xlabel("market seconds"); ax.grid(alpha=.3)
axes[1].axhline(0.5, ls="--", c="k", lw=1)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 6. Export the winner for molab
!ls -la models/best.pt
# Download models/best.pt, drop it into the molab workspace, then open
# notebooks/molab_live.py with `marimo edit`.